# Forecasting S&P 500 Volatility: Machine Learning vs. HAR-RV and GARCH

Forecasts the realized volatility of the S&P 500 over the next 5 trading days and compares
7 machine learning models against the standard benchmarks from the volatility literature.

**Run order:** run all cells top to bottom (about 20-25 minutes on a Kaggle CPU session).

> On Kaggle, turn **Internet** on in the session options before running.

## 1. Download S&P 500 and VIX daily data

In [ ]:
!pip install -q yfinance arch

import os
import time

import numpy as np
import pandas as pd
import yfinance as yf

START_DATE = "2000-01-01"
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)


def fetch_history(ticker, start, max_retries=3, delay=5):
    """Download daily OHLCV history with retries to handle yfinance rate limits."""
    for attempt in range(1, max_retries + 1):
        try:
            hist = yf.Ticker(ticker).history(start=start, auto_adjust=False)
            if not hist.empty:
                hist.index = hist.index.tz_localize(None).normalize()
                return hist
        except Exception as e:
            print(f"{ticker}: attempt {attempt} failed ({e})")
        time.sleep(delay)
    raise RuntimeError(f"Could not download data for {ticker}")


spx = fetch_history("^GSPC", START_DATE)
time.sleep(1)  # avoid yfinance rate limiting
vix = fetch_history("^VIX", START_DATE)

# Keep only trading days that exist in both series
df = spx[["Open", "High", "Low", "Close", "Volume"]].join(
    vix["Close"].rename("VIX"), how="inner"
)
df.index.name = "Date"
df.to_csv(f"{DATA_DIR}/spx_vix_daily.csv")

print(f"Rows: {len(df)}")
print(f"Date range: {df.index.min().date()} -> {df.index.max().date()}")
print("\nMissing values per column:")
print(df.isna().sum())

## 2. Build features and target

Every feature uses information available at the **close of day t**; the target uses
days **t+1 to t+5** only, so features never see the future.

Yahoo Finance reports zero volume for one day (2023-05-24); it is filled with the previous
day's volume so the series stays continuous.

In [ ]:
HORIZON = 5          # forecast horizon in trading days
ANNUALIZE = 252      # trading days per year

feat = pd.DataFrame(index=df.index)

# Daily log return and squared return (daily variance proxy)
log_ret = np.log(df["Close"] / df["Close"].shift(1))
sq_ret = log_ret ** 2


def realized_vol(sq, window):
    """Annualized realized volatility (%) from the mean of squared returns over a window."""
    return np.sqrt(ANNUALIZE * sq.rolling(window).mean()) * 100


# --- HAR features: daily, weekly and monthly realized volatility ---
feat["rv_d"] = realized_vol(sq_ret, 1)
feat["rv_w"] = realized_vol(sq_ret, 5)
feat["rv_m"] = realized_vol(sq_ret, 22)

# --- Range-based (Parkinson) volatility, a less noisy daily estimator ---
park_var = (np.log(df["High"] / df["Low"]) ** 2) / (4 * np.log(2))
feat["park_d"] = np.sqrt(ANNUALIZE * park_var) * 100
feat["park_w"] = np.sqrt(ANNUALIZE * park_var.rolling(5).mean()) * 100

# --- Market fear gauge ---
feat["vix"] = df["VIX"]
feat["vix_change"] = np.log(df["VIX"] / df["VIX"].shift(1))

# --- Past returns (falling markets tend to raise volatility: leverage effect) ---
feat["ret_d"] = log_ret * 100
feat["ret_w"] = np.log(df["Close"] / df["Close"].shift(5)) * 100

# --- Relative trading volume vs. its 22-day average (zero-volume days filled forward) ---
print(f"Zero-volume days in raw data: {(df['Volume'] == 0).sum()}")
volume = df["Volume"].replace(0, np.nan).ffill()
feat["volume_ratio"] = np.log(volume / volume.rolling(22).mean())

# --- Target: realized volatility over the NEXT 5 trading days (t+1..t+5) ---
future_sq = sum(sq_ret.shift(-i) for i in range(1, HORIZON + 1))
feat["target_rv_5d"] = np.sqrt(ANNUALIZE * future_sq / HORIZON) * 100

# Keep the daily return for GARCH, then drop warm-up rows and the last 5 rows (no future yet)
feat["log_ret"] = log_ret
data = feat.replace([np.inf, -np.inf], np.nan).dropna().copy()
data.to_csv(f"{DATA_DIR}/features.csv")

FEATURES = [c for c in data.columns if c not in ("target_rv_5d", "log_ret")]
print(f"Rows: {len(data)}")
print(f"Date range: {data.index.min().date()} -> {data.index.max().date()}")
print(f"Features ({len(FEATURES)}): {FEATURES}")

# The series must be continuous (important for LSTM sequences)
expected_rows = df.loc[data.index.min():data.index.max()].shape[0]
print(f"Gaps in the series: {expected_rows - len(data)}")

print("\nTarget summary (annualized %):")
print(data["target_rv_5d"].describe().round(2))

print("\nCorrelation of each feature with the target:")
print(data[FEATURES].corrwith(data["target_rv_5d"]).sort_values(ascending=False).round(3))

## 3. Train/test split, evaluation metrics and naive baseline

- **Test:** the last 504 trading days (~2 years), never used for tuning.
- **Embargo:** 5 days between train and test, so the last training targets do not overlap the test period.
- **Metrics:** RMSE, MAE and QLIKE (the standard volatility loss, which penalizes under-prediction more).

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

TEST_DAYS = 504   # ~2 trading years held out for the final test
EMBARGO = HORIZON # gap so training targets never overlap the test period

TARGET = "target_rv_5d"
test = data.iloc[-TEST_DAYS:]
train = data.iloc[: -(TEST_DAYS + EMBARGO)]

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

print(f"Train: {train.index.min().date()} -> {train.index.max().date()} ({len(train)} rows)")
print(f"Test:  {test.index.min().date()} -> {test.index.max().date()} ({len(test)} rows)")
print(f"\nTarget mean  | train: {y_train.mean():.2f}  test: {y_test.mean():.2f}")
print(f"Target max   | train: {y_train.max():.2f}  test: {y_test.max():.2f}")


def qlike(y_true, y_pred):
    """QLIKE loss on variances: penalizes under-prediction of volatility more than over-prediction."""
    ratio = (y_true ** 2) / (np.maximum(y_pred, 1e-6) ** 2)
    return np.mean(ratio - np.log(ratio) - 1)


def evaluate(name, y_true, y_pred):
    """Return a dict of evaluation metrics for one model."""
    return {
        "model": name,
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "QLIKE": qlike(np.asarray(y_true), np.asarray(y_pred)),
    }


# Collect test predictions and scores of every model in one place
predictions = pd.DataFrame({"actual": y_test})
results = []

# Baseline 1: naive forecast = last week's realized volatility
predictions["Naive"] = X_test["rv_w"]
results.append(evaluate("Naive", y_test, predictions["Naive"]))

display(pd.DataFrame(results).set_index("model").round(3))

## 4. Literature baselines: HAR-RV and GARCH(1,1)

GARCH parameters are estimated on the training period only, then fixed and filtered through
the test period, so each forecast at day t only uses returns up to day t.

In [ ]:
from arch import arch_model
from sklearn.linear_model import LinearRegression

# --- Baseline 2: HAR-RV (linear regression on daily, weekly, monthly realized volatility) ---
HAR_FEATURES = ["rv_d", "rv_w", "rv_m"]
har = LinearRegression().fit(X_train[HAR_FEATURES], y_train)
predictions["HAR-RV"] = har.predict(X_test[HAR_FEATURES])
results.append(evaluate("HAR-RV", y_test, predictions["HAR-RV"]))

print("HAR-RV coefficients:")
for name, coef in zip(HAR_FEATURES, har.coef_):
    print(f"  {name}: {coef:.3f}")
print(f"  intercept: {har.intercept_:.3f}")

# --- Baseline 3: GARCH(1,1) with Student-t errors ---
returns_pct = data["log_ret"] * 100  # arch works best with returns in percent

# 1) Estimate parameters on the training period only
garch_train = arch_model(returns_pct.loc[: train.index.max()], mean="Constant",
                         vol="GARCH", p=1, q=1, dist="t")
garch_fit = garch_train.fit(disp="off")
print("\nGARCH(1,1) parameters (estimated on train only):")
print(garch_fit.params.round(4))

# 2) Fix those parameters and filter through the full series
garch_full = arch_model(returns_pct, mean="Constant", vol="GARCH", p=1, q=1, dist="t")
garch_fixed = garch_full.fix(garch_fit.params)
forecast = garch_fixed.forecast(horizon=HORIZON, start=test.index[0], reindex=False)

# 3) Convert the 5 daily variance forecasts into annualized 5-day volatility (%)
garch_vol = np.sqrt(ANNUALIZE * forecast.variance.mean(axis=1))
predictions["GARCH"] = garch_vol.reindex(test.index)
print(f"\nGARCH forecasts missing: {predictions['GARCH'].isna().sum()}")
results.append(evaluate("GARCH", y_test, predictions["GARCH"]))

display(pd.DataFrame(results).set_index("model").round(3))

## 5. Machine learning models with time-series grid search

- `TimeSeriesSplit` with 5 folds and a 5-day gap: each fold trains on the past and validates on the period after it.
- Scaling sits inside a `Pipeline`, so it is fitted on each training fold only.
- The test period is never used during tuning.

In [ ]:
import time as timer

from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from xgboost import XGBRegressor

SEED = 42

# Each fold trains on the past and validates on the following period;
# the gap stops 5-day targets from overlapping the validation fold
tscv = TimeSeriesSplit(n_splits=5, gap=HORIZON)

MODELS = {
    "Ridge": (
        Pipeline([("scaler", StandardScaler()), ("model", Ridge())]),
        {"model__alpha": [0.01, 0.1, 1, 10, 100]},
    ),
    "SVR": (
        Pipeline([("scaler", StandardScaler()), ("model", SVR(kernel="rbf"))]),
        {"model__C": [1, 10, 100], "model__epsilon": [0.1, 0.5, 1.0], "model__gamma": ["scale", 0.01]},
    ),
    "RandomForest": (
        RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1),
        {"max_depth": [4, 6, 10, None], "min_samples_leaf": [5, 20, 50], "max_features": [0.5, 1.0]},
    ),
    "XGBoost": (
        XGBRegressor(subsample=0.8, random_state=SEED, n_jobs=-1),
        {"n_estimators": [300, 600], "max_depth": [2, 3, 5],
         "learning_rate": [0.02, 0.05], "colsample_bytree": [0.8, 1.0]},
    ),
    "LightGBM": (
        LGBMRegressor(random_state=SEED, verbose=-1),
        {"n_estimators": [300, 600], "num_leaves": [7, 15, 31],
         "learning_rate": [0.02, 0.05], "min_child_samples": [20, 50]},
    ),
    "MLP": (
        Pipeline([("scaler", StandardScaler()),
                  ("model", MLPRegressor(max_iter=500, random_state=SEED))]),
        {"model__hidden_layer_sizes": [(32,), (64, 32)], "model__alpha": [1e-4, 1e-3, 1e-2]},
    ),
}

best_models, cv_summary = {}, []
for name, (estimator, grid) in MODELS.items():
    start = timer.time()
    search = GridSearchCV(estimator, grid, cv=tscv,
                          scoring="neg_root_mean_squared_error", n_jobs=-1)
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_

    predictions[name] = search.best_estimator_.predict(X_test)
    results.append(evaluate(name, y_test, predictions[name]))
    cv_summary.append({
        "model": name,
        "CV_RMSE": -search.best_score_,
        "combinations": len(search.cv_results_["params"]),
        "minutes": (timer.time() - start) / 60,
        "best_params": search.best_params_,
    })
    print(f"{name}: CV RMSE {-search.best_score_:.3f} | {cv_summary[-1]['minutes']:.1f} min")

print("\nBest parameters per model:")
for row in cv_summary:
    print(f"  {row['model']}: {row['best_params']}")

### 5b. Extend the grids where the best value hit the grid edge

In the first pass, most models picked a value at the edge of their grid, so the true optimum
could lie outside it. This second pass extends those grids and reports any edges that remain.

In [ ]:
REFINED = {
    "Ridge": (
        Pipeline([("scaler", StandardScaler()), ("model", Ridge())]),
        {"model__alpha": [100, 300, 1000, 3000, 10000]},
    ),
    "SVR": (
        Pipeline([("scaler", StandardScaler()), ("model", SVR(kernel="rbf"))]),
        {"model__C": [100, 300, 1000], "model__epsilon": [1.0, 2.0, 3.0],
         "model__gamma": [0.001, 0.003, 0.01]},
    ),
    "XGBoost": (
        XGBRegressor(subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=-1),
        {"max_depth": [1, 2], "learning_rate": [0.005, 0.01, 0.02],
         "n_estimators": [600, 1200, 2000]},
    ),
    "LightGBM": (
        LGBMRegressor(min_child_samples=20, random_state=SEED, verbose=-1),
        {"num_leaves": [3, 5, 7], "learning_rate": [0.005, 0.01, 0.02],
         "n_estimators": [300, 600, 1200]},
    ),
    "MLP": (
        Pipeline([("scaler", StandardScaler()),
                  ("model", MLPRegressor(max_iter=2000, random_state=SEED))]),
        {"model__hidden_layer_sizes": [(16,), (32,)], "model__alpha": [1e-3, 1e-2, 1e-1]},
    ),
}


def edge_params(best_params, grid):
    """List numeric parameters whose best value is the smallest or largest value in the grid."""
    edges = []
    for key, values in grid.items():
        numeric = [v for v in values if isinstance(v, (int, float))]
        if len(numeric) > 1 and best_params[key] in (min(numeric), max(numeric)):
            edges.append(f"{key}={best_params[key]}")
    return edges


# Drop the first-pass scores of the models being re-tuned
results = [r for r in results if r["model"] not in REFINED]

for name, (estimator, grid) in REFINED.items():
    start = timer.time()
    search = GridSearchCV(estimator, grid, cv=tscv,
                          scoring="neg_root_mean_squared_error", n_jobs=-1)
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_
    predictions[name] = search.best_estimator_.predict(X_test)
    results.append(evaluate(name, y_test, predictions[name]))

    edges = edge_params(search.best_params_, grid)
    print(f"{name}: CV RMSE {-search.best_score_:.3f} | {(timer.time() - start) / 60:.1f} min")
    print(f"   best: {search.best_params_}")
    print(f"   still on grid edge: {edges or 'none'}")

## 6. LSTM on 22-day sequences

The best configuration is chosen on a time-based validation split (last 20% of the training
period, with a 5-day gap), never on the test period.

In [ ]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TensorFlow info logs

import tensorflow as tf
from tensorflow import keras

WINDOW = 22  # one trading month of history per sample

# Scale features and target using training-period statistics only
seq_scaler = StandardScaler().fit(X_train)
X_all = seq_scaler.transform(data[FEATURES])
y_mean, y_std = y_train.mean(), y_train.std()
y_all = ((data[TARGET] - y_mean) / y_std).values


def make_sequences(end_positions):
    """Build (samples, WINDOW, features) arrays ending at each row position (inclusive)."""
    X = np.stack([X_all[p - WINDOW + 1: p + 1] for p in end_positions])
    return X, y_all[end_positions]


# Row positions: each sample is labelled by its last day
train_pos = np.arange(WINDOW - 1, len(train))
test_pos = np.arange(len(data) - TEST_DAYS, len(data))

# Time-based validation: last 20% of training samples, with a gap so targets do not overlap
n_val = int(0.2 * len(train_pos))
fit_pos, val_pos = train_pos[: -(n_val + HORIZON)], train_pos[-n_val:]

X_fit, y_fit = make_sequences(fit_pos)
X_val, y_val = make_sequences(val_pos)
X_seq_test, _ = make_sequences(test_pos)
print(f"Sequences | fit: {X_fit.shape} | val: {X_val.shape} | test: {X_seq_test.shape}")


def build_lstm(units, dropout):
    """Single-layer LSTM followed by a small dense head."""
    model = keras.Sequential([
        keras.layers.Input(shape=(WINDOW, len(FEATURES))),
        keras.layers.LSTM(units),
        keras.layers.Dropout(dropout),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dense(1),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse")
    return model


LSTM_GRID = [{"units": u, "dropout": d} for u in (16, 32) for d in (0.1, 0.3)]
best_val, best_lstm, best_cfg = np.inf, None, None

for cfg in LSTM_GRID:
    keras.utils.set_random_seed(SEED)
    model = build_lstm(**cfg)
    stop = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
    history = model.fit(X_fit, y_fit, validation_data=(X_val, y_val), epochs=100,
                        batch_size=64, callbacks=[stop], verbose=0)
    # Validation RMSE in the original units (annualized %)
    val_pred = model.predict(X_val, verbose=0).ravel() * y_std + y_mean
    val_rmse = np.sqrt(mean_squared_error(y_val * y_std + y_mean, val_pred))
    print(f"{cfg}: val RMSE {val_rmse:.3f} | epochs {len(history.history['loss'])}")
    if val_rmse < best_val:
        best_val, best_lstm, best_cfg = val_rmse, model, cfg

print(f"\nBest LSTM config: {best_cfg} (val RMSE {best_val:.3f})")

predictions["LSTM"] = best_lstm.predict(X_seq_test, verbose=0).ravel() * y_std + y_mean
results.append(evaluate("LSTM", y_test, predictions["LSTM"]))

order = ["Naive", "HAR-RV", "GARCH", "Ridge", "SVR", "RandomForest",
         "XGBoost", "LightGBM", "MLP", "LSTM"]
results_df = pd.DataFrame(results).set_index("model").loc[order]
print("\nTest results (all models):")
display(results_df.round(3))

## 7. Diebold-Mariano tests

Are the differences between models statistically significant? Uses a HAC (Newey-West)
variance with h-1 lags, because 5-day targets overlap, plus the Harvey-Leybourne-Newbold
small-sample correction. A **negative statistic** means the model has lower loss than the benchmark.

In [ ]:
from scipy import stats


def squared_error(y_true, y_pred):
    return (np.asarray(y_true) - np.asarray(y_pred)) ** 2


def qlike_loss(y_true, y_pred):
    ratio = np.asarray(y_true) ** 2 / np.maximum(np.asarray(y_pred), 1e-6) ** 2
    return ratio - np.log(ratio) - 1


def dm_test(loss_model, loss_benchmark, h=HORIZON):
    """
    Diebold-Mariano test of equal predictive accuracy.
    Negative statistic -> the model has lower loss than the benchmark.
    Uses h-1 autocovariance lags because 5-day targets overlap.
    """
    d = np.asarray(loss_model) - np.asarray(loss_benchmark)
    T, d_mean = len(d), d.mean()
    gamma = [np.mean((d[k:] - d_mean) * (d[: T - k] - d_mean)) for k in range(h)]
    variance = gamma[0] + 2 * sum(gamma[1:])
    if variance <= 0:  # rare: fall back to the plain variance
        variance = gamma[0]
    dm = d_mean / np.sqrt(variance / T)
    hln = np.sqrt((T + 1 - 2 * h + h * (h - 1) / T) / T)  # small-sample correction
    stat = dm * hln
    p_value = 2 * stats.t.sf(abs(stat), df=T - 1)
    return stat, p_value


y = predictions["actual"]
rows = []
for model in order:
    row = {"model": model}
    for bench in ["HAR-RV", "Ridge"]:
        if model == bench:
            continue
        for loss_name, loss_fn in [("SE", squared_error), ("QLIKE", qlike_loss)]:
            stat, p = dm_test(loss_fn(y, predictions[model]), loss_fn(y, predictions[bench]))
            row[f"vs {bench} {loss_name} stat"] = round(stat, 2)
            row[f"vs {bench} {loss_name} p"] = round(p, 4)
    rows.append(row)

dm_df = pd.DataFrame(rows).set_index("model")
pd.set_option("display.max_columns", None)
display(dm_df)

print("\nSignificantly better than HAR-RV (p < 0.05, negative statistic):")
for loss_name in ["SE", "QLIKE"]:
    better = dm_df[(dm_df[f"vs HAR-RV {loss_name} stat"] < 0) &
                   (dm_df[f"vs HAR-RV {loss_name} p"] < 0.05)].index.tolist()
    print(f"  {loss_name}: {better or 'none'}")

## 8. Feature ablation: where does the improvement come from?

The model is kept fixed (tuned Ridge and XGBoost) while the feature set changes.

In [ ]:
HAR = ["rv_d", "rv_w", "rv_m"]
FEATURE_SETS = {
    "A: HAR only": HAR,
    "B: HAR + VIX": HAR + ["vix", "vix_change"],
    "C: HAR + Parkinson": HAR + ["park_d", "park_w"],
    "D: HAR + VIX + Parkinson": HAR + ["vix", "vix_change", "park_d", "park_w"],
    "E: All features": FEATURES,
}

# Reuse the tuned hyperparameters from the grid searches
ABLATION_MODELS = {
    "Ridge": lambda: Pipeline([("scaler", StandardScaler()),
                               ("model", Ridge(alpha=best_models["Ridge"].named_steps["model"].alpha))]),
    "XGBoost": lambda: XGBRegressor(**{k: v for k, v in best_models["XGBoost"].get_params().items()
                                       if k in ("max_depth", "learning_rate", "n_estimators",
                                                "subsample", "colsample_bytree", "random_state")}),
}

ablation_rows = []
for set_name, cols in FEATURE_SETS.items():
    for model_name, make_model in ABLATION_MODELS.items():
        model = make_model().fit(X_train[cols], y_train)
        pred = model.predict(X_test[cols])
        scores = evaluate(model_name, y_test, pred)
        ablation_rows.append({"features": set_name, "model": model_name,
                              "n_features": len(cols), "RMSE": scores["RMSE"],
                              "MAE": scores["MAE"], "QLIKE": scores["QLIKE"]})

ablation_df = pd.DataFrame(ablation_rows)
print("Reference HAR-RV (plain linear regression): "
      f"RMSE {results_df.loc['HAR-RV', 'RMSE']:.3f} | QLIKE {results_df.loc['HAR-RV', 'QLIKE']:.3f}\n")
display(ablation_df.pivot(index="features", columns="model", values=["RMSE", "QLIKE"]).round(3))

## 9. Figures and saved results

In [ ]:
import matplotlib.pyplot as plt

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# --- Figure 1: actual vs. forecasts over the whole test period ---
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(predictions.index, predictions["actual"], color="black", lw=1.2, label="Actual")
ax.plot(predictions.index, predictions["HAR-RV"], color="#9e9e9e", lw=1, label="HAR-RV")
ax.plot(predictions.index, predictions["Ridge"], color="#1f77b4", lw=1, label="Ridge")
ax.plot(predictions.index, predictions["XGBoost"], color="#d62728", lw=1, label="XGBoost")
ax.set_title("S&P 500: next-5-day realized volatility, test period")
ax.set_ylabel("Annualized volatility (%)")
ax.legend(frameon=False, ncol=4)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/forecasts_test_period.png")
plt.show()

# --- Figure 2: zoom on the largest volatility spike in the test period ---
peak = predictions["actual"].idxmax()
window = predictions.loc[peak - pd.Timedelta(days=45): peak + pd.Timedelta(days=45)]
fig, ax = plt.subplots(figsize=(12, 4.5))
for col, color in [("actual", "black"), ("HAR-RV", "#9e9e9e"), ("GARCH", "#8c564b"),
                   ("Ridge", "#1f77b4"), ("XGBoost", "#d62728")]:
    ax.plot(window.index, window[col], color=color, lw=1.4 if col == "actual" else 1,
            marker="o" if col == "actual" else None, ms=3, label=col.capitalize() if col == "actual" else col)
ax.set_title(f"Zoom on the largest spike (peak on {peak.date()})")
ax.set_ylabel("Annualized volatility (%)")
ax.legend(frameon=False, ncol=5)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/forecasts_spike_zoom.png")
plt.show()

# --- Figure 3: test RMSE and QLIKE for every model ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
baseline_models = {"Naive", "HAR-RV", "GARCH"}
for ax, metric in zip(axes, ["RMSE", "QLIKE"]):
    values = results_df[metric]
    colors = ["#9e9e9e" if m in baseline_models else "#1f77b4" for m in values.index]
    ax.barh(values.index, values.values, color=colors)
    ax.invert_yaxis()
    ax.set_title(f"Test {metric} (lower is better)")
    for i, v in enumerate(values.values):
        ax.text(v, i, f" {v:.3f}", va="center", fontsize=9)
fig.suptitle("Grey = baselines, blue = machine learning models")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/model_comparison.png")
plt.show()

# --- Save every result table ---
predictions.to_csv(f"{DATA_DIR}/test_predictions.csv")
results_df.to_csv(f"{DATA_DIR}/test_results.csv")
dm_df.to_csv(f"{DATA_DIR}/diebold_mariano.csv")
ablation_df.to_csv(f"{DATA_DIR}/ablation.csv", index=False)
pd.DataFrame(cv_summary).to_csv(f"{DATA_DIR}/grid_search_first_pass.csv", index=False)

print("Saved files:")
for folder in [DATA_DIR, FIG_DIR]:
    for name in sorted(os.listdir(folder)):
        print(f"  {folder}/{name} ({os.path.getsize(f'{folder}/{name}') / 1024:.1f} KB)")